In [2]:
import os,warnings,time
warnings.filterwarnings("ignore")

import logfire 
from dotenv import load_dotenv

load_dotenv()

def check_env_var(var_name):
    value = os.getenv(var_name)
    return value is not None and value.strip() != ''

def print_status():
    variables = {
        'LOG_FIRE_TOKEN': 'LogFire Token',
        'GROQ_API_KEY': 'Groq API Key',
        'GOOGLE_API_KEY': 'Gemini API Key'
    }
    
    print("\033[1;34m🔑 Environment Variables Status\033[0m")
    print("-" * 40)
    for var, desc in variables.items():
        status = check_env_var(var)
        if status:
            print(f"\033[92m✅ {desc:20} : Set\033[0m")
        else:
            print(f"\033[91m❌ {desc:20} : Not Set\033[0m")

if __name__ == "__main__":
    print_status()

🔑 Environment Variables Status
----------------------------------------
✅ LogFire Token        : Set
✅ Groq API Key         : Set
✅ Gemini API Key       : Set


In [3]:
import logfire 

logfire.configure()
logfire.info('Hello,  {place}!', place = 'World')

Logfire project URL: https://logfire-eu.pydantic.dev/mustafakocaman/pydantic-logfire-training

01:01:39.754 Hello,  World!


In [4]:
logfire.configure(
    token = os.getenv("LOG_FIRE_TOKEN"),
    service_name="llm-observability"
)

Logfire project URL: https://logfire-eu.pydantic.dev/mustafakocaman/pydantic-logfire-training

## Basic Info

In [5]:
logfire.info("training_started",
            part = "Part 1 - First",
            instructer = "Mustafa",
            tool = "Pydantic Logfire"

            )

01:07:39.334 training_started


## Trace

In [7]:
with logfire.span("data_processing_simulation", dataset="observability_data", rows=1000):
    logfire.info("step_started", step=1, action="loading data")
    time.sleep(0.3)

    logfire.info("step_started", step=2, action="transforming", columns=12)
    time.sleep(0.2)

    logfire.info("step_started", step=3, action="saving results", output="/tmp/out.csv")

01:14:47.665 data_processing_simulation
01:14:47.667   step_started
01:14:47.968   step_started
01:14:48.170   step_started


## Pydantic Validation

In [8]:
from pydantic import BaseModel
from typing import Optional

# Mock Data

class LLMRequest(BaseModel):
    user_id: str
    session_id: str
    query: str
    model: str
    temperature: float = 0.7
    max_tokens: Optional[int] = None


class LLMResponse(BaseModel):
    answer: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    model_used: str    



In [11]:
import time
import logfire
from pydantic import BaseModel


class LLMRequest(BaseModel):
    user_id: str
    session_id: str
    query: str
    model: str
    max_tokens: int

class LLMResponse(BaseModel):
    answer: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    model_used: str

def pretty_print_response(resp):
    data = resp.model_dump()
    print("\033[1;94m" + "┌" + "─"*48 + "┐" + "\033[0m")
    print("\033[1;94m│" + " 🤖 LLM Response Details".ljust(48) + "│\033[0m")
    print("\033[1;94m├" + "─"*48 + "┤\033[0m")
    print(f"\033[92m│ 📝 Answer    : {str(data.get('answer', ''))[:35]}...".ljust(49) + "│\033[0m")
    print(f"\033[93m│ 🔢 Input Tok: {data.get('input_tokens')}".ljust(49) + "│\033[0m")
    print(f"\033[93m│ 🔢 Output Tok: {data.get('output_tokens')}".ljust(49) + "│\033[0m")
    print(f"\033[94m│ ⏱️ Latency  : {data.get('latency_ms')} ms".ljust(49) + "│\033[0m")
    print(f"\033[95m│ 🤖 Model    : {data.get('model_used')}".ljust(49) + "│\033[0m")
    print("\033[1;94m└" + "─"*48 + "┘\033[0m")


request = LLMRequest(
    user_id="Mustafa",
    session_id="session_35",
    query="What is RAG?",
    model="llama-3.3-70b-versatile",
    max_tokens=500
)

with logfire.span("llm_CALL",
                  user_id = request.user_id,
                  session_id = request.session_id,
                  model_used = request.model):
    logfire.info("request_received" , **request.model_dump())
    
    time.sleep(0.1)

    response = LLMResponse(
        answer="RAG is a technique that retrieves relevant documents...",
        input_tokens=18,
        output_tokens=120,
        latency_ms=342.5,
        model_used="llama-3.3-70b-versatile"
    )
    logfire.info("response_sent", **response.model_dump())

pretty_print_response(response)

01:47:17.448 llm_CALL
01:47:17.449   request_received
01:47:17.551   response_sent
┌────────────────────────────────────────────────┐
│ 🤖 LLM Response Details                         │
├────────────────────────────────────────────────┤
│ 📝 Answer    : RAG is a technique that retrieves r...│
│ 🔢 Input Tok: 18                           │
│ 🔢 Output Tok: 120                         │
│ ⏱️ Latency  : 342.5 ms                    │
│ 🤖 Model    : llama-3.3-70b-versatile      │
└────────────────────────────────────────────────┘


## Groq Instrumentation

In [13]:
import textwrap
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


logfire.instrument_openai()

llm_groq = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.3-70b-versatile",
    temperature=0.3
)


def pretty_print_response(text):
    width = 60
    print("\033[1;96m┌" + "─" * width + "┐\033[0m")
    print("\033[1;96m│" + " 🤖 Groq (llama-3.3) Response".ljust(width) + "│\033[0m")
    print("\033[1;96m├" + "─" * width + "┤\033[0m")
    
    # wrap
    lines = textwrap.wrap(text, width=width-4) 
    for line in lines:
        print(f"\033[97m│ {line:<{width-1}}│\033[0m")
    print("\033[1;96m└" + "─" * width + "┘\033[0m")

# Make a call — watch the trace appear in the dashboard automatically
print("Calling Groq (llama-3.3-70b)…")
response = llm_groq.invoke([
    HumanMessage(content="Explain why kitchen sponges smell bad and how to prevent it, in 3 bullet points.")
])


pretty_print_response(response.content)

Calling Groq (llama-3.3-70b)…
02:03:08.194 Chat Completion with 'llama-3.3-70b-versatile' [LLM]
┌────────────────────────────────────────────────────────────┐
│ 🤖 Groq (llama-3.3) Response                                │
├────────────────────────────────────────────────────────────┤
│ Here are 3 bullet points explaining why kitchen sponges    │
│ smell bad and how to prevent it:  * Kitchen sponges        │
│ smell bad because they provide a warm, moist environment   │
│ that is ideal for the growth of bacteria, mold, and        │
│ mildew. These microorganisms feed on food particles and    │
│ other organic matter that get trapped in the sponge,       │
│ causing it to emit unpleasant odors. * To prevent          │
│ kitchen sponges from smelling bad, it's essential to       │
│ regularly sanitize them. This can be done by microwaving   │
│ the sponge for 2 minutes or running it through the         │
│ dishwasher. Sanitizing the sponge kills bacteria and       │
│ other microorganisms

## Gemini Instrumentation

In [16]:
def pretty_print_response(text):
    width = 60
    print("\033[1;96m┌" + "─" * width + "┐\033[0m")
    print("\033[1;96m│" + " 🤖 Gemini (gemini-2.5-flash) Response".ljust(width) + "│\033[0m")
    print("\033[1;96m├" + "─" * width + "┤\033[0m")
    
    wrapped_lines = textwrap.wrap(text, width=width-4)
    for line in wrapped_lines:
        print(f"\033[97m│ {line:<{width-1}}│\033[0m")
    print("\033[1;96m└" + "─" * width + "┘\033[0m")

# ---------- Instrumentation (optional, but keeps trace) ----------
logfire.instrument_openai()

# ---------- Gemini setup (unchanged) ----------
llm_gemini = ChatOpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GOOGLE_API_KEY"),
    model="gemini-2.5-flash",
    temperature=0.3
)

# ---------- Call with a question ----------
print("Calling Gemini (gemini-2.5-flash)…")
try:
    response = llm_gemini.invoke([
        HumanMessage(content="Why do bananas turn brown quickly? Explain in 2 short sentences.")
    ])

    pretty_print_response(response.content)
except Exception as e:
    print(f"\033[91m⚠️  Gemini call failed: {e}\033[0m")
    print("    Check your GOOGLE_API_KEY in .env")

Calling Gemini (gemini-2.5-flash)…
02:10:27.284 Chat Completion with 'gemini-2.5-flash' [LLM]
┌────────────────────────────────────────────────────────────┐
│ 🤖 Gemini (gemini-2.5-flash) Response                       │
├────────────────────────────────────────────────────────────┤
│ Bananas turn brown quickly due to enzymes that react       │
│ with oxygen in the air. This process, called oxidation,    │
│ breaks down their starches and sugars, causing the skin    │
│ and flesh to darken.                                       │
└────────────────────────────────────────────────────────────┘


## Compare Models

In [ ]:
query = "What is the difference between RAG and fine-tuning? Give 3 bullet points."

with logfire.span("model_comparison", query=query, num_models=2):

    # ── Groq ─────────────────────────────────────────────────────────────
    with logfire.span("groq_call", model="llama-3.3-70b-versatile", provider="groq"):
        t0 = time.time()
        r_groq = llm_groq.invoke([HumanMessage(content=query)])
        groq_ms = round((time.time() - t0) * 1000, 1)
        logfire.info("groq_done", latency_ms=groq_ms, answer_len=len(r_groq.content))

    # ── Gemini ────────────────────────────────────────────────────────────
    with logfire.span("gemini_call", model="gemini-2.5-flash", provider="google"):
        t0 = time.time()
        try:
            r_gemini = llm_gemini.invoke([HumanMessage(content=query)])
            gemini_ms = round((time.time() - t0) * 1000, 1)
            logfire.info("gemini_done", latency_ms=gemini_ms, answer_len=len(r_gemini.content))
            gemini_answer = r_gemini.content
        except Exception as e:
            logfire.warning("gemini_failed", error=str(e))
            gemini_ms = 0
            gemini_answer = f"[Error: {e}]"

# ── Print results ─────────────────────────────────────────────────────────
print(f"🟢 Groq ({groq_ms}ms):\n{r_groq.content}")
print(f"\n🔵 Gemini ({gemini_ms}ms):\n{gemini_answer}")

In [18]:
def pretty_print_comparison(groq_text, groq_ms, gemini_text, gemini_ms):
    width = 70
    print("\033[1;96m┌" + "─" * width + "┐\033[0m")
    print("\033[1;96m│" + " 🤖 Model Comparison (Groq vs Gemini)".ljust(width) + "│\033[0m")
    print("\033[1;96m├" + "─" * width + "┤\033[0m")
    
    # Groq section
    print("\033[1;92m│ 🟢 GROQ".ljust(width) + "│\033[0m")
    print("\033[92m│ ⏱️  Time: {} ms".format(groq_ms).ljust(width) + "│\033[0m")
    print("\033[92m├" + "─" * width + "┤\033[0m")
    groq_lines = textwrap.wrap(groq_text, width=width-4)
    for line in groq_lines:
        print(f"\033[97m│ {line:<{width-1}}│\033[0m")
    
    # Gemini section
    print("\033[1;94m├" + "─" * width + "┤\033[0m")
    print("\033[1;94m│ 🔵 GEMINI".ljust(width) + "│\033[0m")
    print("\033[94m│ ⏱️  Time: {} ms".format(gemini_ms).ljust(width) + "│\033[0m")
    print("\033[94m├" + "─" * width + "┤\033[0m")
    gemini_lines = textwrap.wrap(gemini_text, width=width-4)
    for line in gemini_lines:
        print(f"\033[97m│ {line:<{width-1}}│\033[0m")
    
    print("\033[1;96m└" + "─" * width + "┘\033[0m")

# Model setup 
logfire.instrument_openai()

llm_groq = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.3-70b-versatile",
    temperature=0.3
)

llm_gemini = ChatOpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GOOGLE_API_KEY"),
    model="gemini-2.5-flash",
    temperature=0.3
)

# ---------- Query ----------
query = "Why do we yawn? Explain in 2 sentences and give 3 possible reasons."


with logfire.span("model_comparison", query=query, num_models=2):

    # ── Groq ─────────────────────────────────────────────────────────────
    with logfire.span("groq_call", model="llama-3.3-70b-versatile", provider="groq"):
        t0 = time.time()
        r_groq = llm_groq.invoke([HumanMessage(content=query)])
        groq_ms = round((time.time() - t0) * 1000, 1)
        logfire.info("groq_done", latency_ms=groq_ms, answer_len=len(r_groq.content))

    # ── Gemini ────────────────────────────────────────────────────────────
    with logfire.span("gemini_call", model="gemini-2.5-flash", provider="google"):
        t0 = time.time()
        try:
            r_gemini = llm_gemini.invoke([HumanMessage(content=query)])
            gemini_ms = round((time.time() - t0) * 1000, 1)
            logfire.info("gemini_done", latency_ms=gemini_ms, answer_len=len(r_gemini.content))
            gemini_answer = r_gemini.content
        except Exception as e:
            logfire.warning("gemini_failed", error=str(e))
            gemini_ms = 0
            gemini_answer = f"[Error: {e}]"

pretty_print_comparison(r_groq.content, groq_ms, gemini_answer, gemini_ms)

02:17:03.331 model_comparison
02:17:03.333   groq_call
02:17:03.336     Chat Completion with 'llama-3.3-70b-versatile' [LLM]
02:17:04.055     groq_done
02:17:04.057   gemini_call
02:17:04.058     Chat Completion with 'gemini-2.5-flash' [LLM]
02:17:08.507     gemini_done
┌──────────────────────────────────────────────────────────────────────┐
│ 🤖 Model Comparison (Groq vs Gemini)                                  │
├──────────────────────────────────────────────────────────────────────┤
│ 🟢 GROQ                                                       │
│ ⏱️  Time: 720.9 ms                                             │
├──────────────────────────────────────────────────────────────────────┤
│ Yawning is a universal and involuntary action that occurs in         │
│ humans and many other animals, and its exact purpose is still not    │
│ fully understood, but it is believed to be related to three          │
│ possible reasons: brain temperature regulation, oxygenation of the   │
│ body, and r

## RAG Pipeline Tracing

### Data Ingestion

In [19]:
import json
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document


# Load knowledge base
with open("documents.json") as f:
    raw_docs = json.load(f)
    
# Convert the docs IN langchain compattipble documents

DOCS = [
    Document(page_content=d["content"], metadata={"topic": d["topic"], "source": d["source"]})
    for d in raw_docs
]
print(f"Loaded {len(DOCS)} documents: {[d.metadata['topic'] for d in DOCS]}")


Loaded 6 documents: ['RAG', 'Guardrails', 'Gateway', 'Observability', 'Evals', 'Fine-tuning']


In [20]:
DOCS

[Document(metadata={'topic': 'RAG', 'source': 'doc_1'}, page_content='Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. When a user asks a question, RAG first retrieves relevant documents from a knowledge base using vector similarity search, then passes those documents along with the question to an LLM. This grounds the answer in actual content, which significantly reduces hallucinations compared to pure LLM generation.'),
 Document(metadata={'topic': 'Guardrails', 'source': 'doc_2'}, page_content='LLM Guardrails are safety controls that sit between the user and the language model. They run before the LLM sees the input (input rails) and after the LLM generates output (output rails). NVIDIA NeMo Guardrails uses a domain-specific language called Colang to define rules declaratively. Common guardrails include prompt injection detection, PII filtering, toxicity filtering, and topic restriction.'),
 Document(metadata={'topic': 'Gateway', 'source': 'd

In [23]:
# Embeddings + FAISS index 
embeddings  = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

vectorstore = FAISS.from_documents(DOCS, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 2})
print("✅  FAISS index ready")
print("✅   Data Ingestion done")



✅  FAISS index ready
✅   Data Ingestion done


### Retrieval Pipeline

In [24]:
# RAG with tracing 
def rag(question: str, user_id: str = "anonymous") -> str:
    with logfire.span("rag_pipeline", question=question, user_id=user_id):
        docs = retriever.invoke(question)   # search similar vectors 
        logfire.info("docs_retrieved",
                     topics=[d.metadata["topic"] for d in docs],
                     num_docs=len(docs))
        context = "\n\n".join(
            f"[{d.metadata['topic']}] {d.page_content}" for d in docs
        )
        prompt = (
            f"Answer the question based only on the context below.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\nAnswer concisely:"
        )
        return llm_groq.invoke(prompt).content


In [25]:
answer = rag("How does a Rag reduce hallucination" , user_id="Mustafa35")
print(f"\nA: {answer}")

02:37:40.743 rag_pipeline
02:37:41.614   docs_retrieved
02:37:41.620   Chat Completion with 'llama-3.3-70b-versatile' [LLM]

A: RAG reduces hallucinations by grounding the answer in actual content from retrieved documents.


## ReAct Agent 

In [26]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage


In [27]:
# Retriever as a plain Python tool
@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for LLM production topics: RAG, guardrails,
    gateways, observability, evaluations, and fine-tuning."""
    docs = vectorstore.similarity_search(query, k=2)
    return "\n\n".join(
        f"[{d.metadata['topic']}] {d.page_content}" for d in docs
    )


In [ ]:
system_prompt = """
You are a senior LLM production engineer and consultant. Your expertise covers 
Retrieval-Augmented Generation (RAG), Guardrails, Gateways, Observability, Evals, 
and Fine‑tuning. You have access to a curated knowledge base via the tool 
`search_knowledge_base`.

### Guidelines
1. **For any question** that includes one of the keywords (RAG, guardrails, gateway, 
   observability, evals, fine‑tuning) or asks about LLM production practices, 
   **always call `search_knowledge_base` first**.

2. **When the KB returns results**:
   - Synthesize the answer clearly and concisely, using **only** the retrieved content.
   - Explicitly cite the source (e.g., `(doc_1)`, `(doc_3)`).
   - If multiple documents are relevant, compare them, highlight differences, or 
     combine insights into a coherent response.
   - For lists, comparisons, or step‑by‑step instructions, format the output 
     accordingly (e.g., numbered bullets, tables if appropriate).

3. **If the KB does not contain relevant information**:
   - Acknowledge that the KB lacks that specific detail, then answer using your 
     general knowledge, but clearly label that part as 'general knowledge'.

4. **For questions completely outside the KB topics** (e.g., 'what is the weather?'), 
   you may answer directly without searching, but keep it brief and helpful.

5. **Always maintain a professional, precise, and educational tone.** When appropriate, 
   offer practical advice, trade‑offs, or best practices for production systems.

### Example Behaviours
- User: 'What is RAG?' → search → respond with definition and grounding explanation, cite doc_1.
- User: 'Compare RAG and fine‑tuning.' → search both → present a balanced comparison, cite both docs.
- User: 'How can I reduce hallucinations?' → search KB (if present) or answer from general knowledge, 
   clearly differentiating source.

Remember: Your primary duty is to deliver accurate, traceable answers using the provided knowledge base 
whenever the topic relates to LLM production systems.
"""

agent = create_agent(
    model=llm_groq,
    tools=[search_knowledge_base],
    system_prompt=system_prompt
)